Problem 4: Naive Bayes classifier

In this problem you will implement your own Naive Bayes classifier and you will compare it with a package implementation. You will use the
Mushroom dataset for this problem. Split the dataset into 75% for training and 25% for testing.

In [60]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn import tree
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_curve, auc
from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

In [61]:
# fetch dataset 
mushroom = fetch_ucirepo(id=73) 
  
# data (as pandas dataframes) 
X = mushroom.data.features 
y = mushroom.data.targets 
  
# metadata 
#print(mushroom.metadata) 
  
# variable information 
#print(mushroom.variables) 

In [62]:
# split data into training and test sets:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=True)


1. Train the Naive Bayes classifier. Compute the prior probabilities for the Edible and Poisonous classes from the training data. For each feature $X_i$ in the dataset compute the probabilities $P[X_i = x| Y=\textit{Edible}]$, and $P[X_i = x| Y=\textit{Poisonous}]$ from the training data. Use the Laplace smoothing method when computing these probabilities. Note that the Naive Bayes classifier stores these prior and conditional probabilities.

In [63]:
# compute probabilities:
# probability of edible given feature and prob of poisonous given feature

# categorical naive bayes

y_train = y_train['poisonous']
X_train = pd.DataFrame(X_train).reset_index(drop=True)
y_train = pd.Series(y_train).reset_index(drop=True)

classes = y_train.unique()
priors = {}
likelihoods = {}
N = len(y_train)

for c in classes:
    X_c = X_train[y_train == c]
    n_c = len(X_c)

    priors[c] = n_c / N

    likelihoods[c] = {}
    
    for feature in X_train.columns:
        k = X_train[feature].nunique()
        likelihoods[c][feature] = {}
        for value in X_train[feature].unique():
            count = (X_c[feature] == value).sum()
            # laplace smoothing: alpha = smoothing parameter
            alpha = 1
            likelihoods[c][feature][value] = (count + alpha) / (n_c + alpha * k)

priors

{'p': 0.48005908419497784, 'e': 0.5199409158050221}

2. For each point in the testing set estimate the probability that it belongs to the Edible and Poisonous classes. Use the Naive Bayes
classifier probabilities computed in part (1).

In [64]:
log_posteriors = {}

for c in classes:
    # start with log prior: log P(Y=c)
    log_prob = np.log(priors[c])

    # add log likelihood for each feature: log P(Xi=x | Y=c)
    for feature, value in X_test.items():
        if value in likelihoods[c][feature]:
            log_prob += np.log(likelihoods[c][feature][value])
        else:
            # unseen value: Laplace smooth with count=0
            k = len(likelihoods[c][feature])
            log_prob += np.log(alpha / (sum(likelihoods[c][feature].values()) + alpha * k))

    log_posteriors[c] = log_prob

    # convert log posteriors back to probabilities using softmax
    max_log  = max(log_posteriors.values())
    exp_vals = {c: np.exp(lp - max_log) for c, lp in log_posteriors.items()}
    total    = sum(exp_vals.values())
    {c: exp_vals[c] / total for c in classes}

def predict_proba(X):
    X = pd.DataFrame(X).reset_index(drop=True)
    probas = [self.predict_proba_single(row) for _, row in X.iterrows()]
    return pd.DataFrame(probas)



proba_df = predict_proba(X_test)
proba_df.columns = ['P(Edible)' if c == 'e' else 'P(Poisonous)' for c in proba_df.columns]
proba_df.index.name = 'Sample'

print("=== Predicted Probabilities for Test Set ===")
display(proba_df.head(10))
print(f"\n{len(proba_df)} total test samples")

TypeError: unhashable type: 'Series'

In [ ]:
def predict(list):
    

3. Compute accuracy, precision, recall, and F1 score for your Naive Bayes classifier on the testing data.

4. Compare the results obtained by your implementation with those obtained with a Naive Bayes package (trained on the same dataset).
Use several metrics, including accuracy, precision, recall, and F1 score. Are the results similar or different?

In [ ]:
X_train_clean = X_train.replace('?', np.nan)
X_test_clean  = X_test.replace('?', np.nan)

for col in X_train_clean.columns:
    mode = X_train_clean[col].mode()[0]
    X_train_clean[col] = X_train_clean[col].fillna(mode)
    X_test_clean[col]  = X_test_clean[col].fillna(mode)

# encode features and labels
encoder = OrdinalEncoder()
X_train_encoded = encoder.fit_transform(X_train_clean)
X_test_encoded  = encoder.transform(X_test_clean)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded  = le.transform(y_test)

model = CategoricalNB(alpha=1)
model.fit(X_train_encoded, y_train_encoded)

/Users/pavithra/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


CategoricalNB(alpha=1)